# Titanic – Family-group model audit
The family-group rule is this project's best model (Kaggle **0.80143**). This notebook challenges its assumptions instead of adding another algorithm:

1. **Family definition.** Does the result depend on how a "family" is built (ticket trimming, fare, port, surname)?
2. **Data quirks.** How many passengers are affected by the edge cases flagged in review (tickets ending in letters, missing port or fare)?
3. **Validation design.** Stratified cross-validation lets relatives sit in both the training and validation folds. **Grouped** cross-validation keeps whole families together, measuring performance on families the rule has never seen.
4. **Test-set stability.** Do different definitions change the same 17 test passengers?

The submitted 0.80143 file is not changed by this notebook.

In [1]:
import glob, numpy as np, pandas as pd, warnings
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold
warnings.filterwarnings('ignore')
def find_file(name):
    for pattern in [f'/kaggle/input/**/{name}', f'../data/{name}', f'data/{name}']:
        hits = glob.glob(pattern, recursive=True)
        if hits: return hits[0]
    raise FileNotFoundError(name)
train_data = pd.read_csv(find_file('train.csv')); test_data = pd.read_csv(find_file('test.csv'))
print(train_data.shape, test_data.shape)

(891, 12) (418, 11)


## Features and data quirks

In [2]:
full = pd.concat([train_data, test_data], ignore_index=True)
full['TitleRaw'] = full['Name'].str.extract(r',\s*([^\.]+)\.')[0]
full['Surname'] = full['Name'].str.split(',').str[0].str.strip()
full['IsBoy'] = full['TitleRaw'] == 'Master'
full['IsWoman'] = full['Sex'] == 'female'
full['WomanOrChild'] = full['IsBoy'] | full['IsWoman']
emb = full['Embarked'].fillna('S'); fare = full['Fare'].round(2).astype(str)
tk_trim = full['Ticket'].str[:-1]
tk_digit = full['Ticket'].str.replace(r'\d$', '', regex=True)  # drop last char only if it's a digit

t = full['Ticket']
print("Tickets ending in a letter:", (~t.str[-1].str.isdigit()).sum(), sorted(t[~t.str[-1].str.isdigit()].unique()))
print("Missing Embarked:", full['Embarked'].isna().sum(), full.loc[full['Embarked'].isna(), ['Name', 'Ticket', 'Fare']].values.tolist())
print("Missing Fare:", full['Fare'].isna().sum(), full.loc[full['Fare'].isna(), ['Name', 'Sex']].values.tolist())
print("Surnames with stray spaces:", (full['Name'].str.split(',').str[0] != full['Surname']).sum())

Tickets ending in a letter: 4 ['LINE']
Missing Embarked: 2 [['Icard, Miss. Amelie', '113572', 80.0], ['Stone, Mrs. George Nelson (Martha Evelyn)', '113572', 80.0]]
Missing Fare: 1 [['Storey, Mr. Thomas', 'male']]
Surnames with stray spaces: 0


## Family definitions
Each definition is a list of keys tried in order; the first key that matches a family in the training data is used. The first one is the definition behind the 0.80143 submission.

In [3]:
DEFS = {
  'Current (surname+class+ticket[:-1]+port+fare, ticket fallback)': [full['Surname']+'-'+full['Pclass'].astype(str)+'-'+tk_trim+'-'+emb+'-'+fare, full['Ticket']],
  'Drop last digit only (not last character)':                      [full['Surname']+'-'+full['Pclass'].astype(str)+'-'+tk_digit+'-'+emb+'-'+fare, full['Ticket']],
  'Without fare':                                                    [full['Surname']+'-'+full['Pclass'].astype(str)+'-'+tk_trim+'-'+emb, full['Ticket']],
  'Without fare or port':                                            [full['Surname']+'-'+full['Pclass'].astype(str)+'-'+tk_trim, full['Ticket']],
  'Surname + exact ticket, ticket fallback':                         [full['Surname']+'-'+full['Ticket'], full['Ticket']],
  'Exact ticket only':                                               [full['Ticket']],
  'Current, no ticket fallback':                                     [full['Surname']+'-'+full['Pclass'].astype(str)+'-'+tk_trim+'-'+emb+'-'+fare],
  'Surname + class only (loose)':                                    [full['Surname']+'-'+full['Pclass'].astype(str)],
}
print(len(DEFS), 'definitions')

8 definitions


## The rule
Start from the gender rule (women survive, men don't). Using only the **women and boys** of each family in the training data: if all of them died, predict that family's women die; if all survived, predict that family's boys survive.

In [4]:
n=len(train_data); y=train_data['Survived'].values
def rule(keys, tr_idx, ap_idx):
    """Gender rule + family overrides; keys = list of key Series over `full`, first match wins."""
    wc_tr = [i for i in tr_idx if full.at[i,'WomanOrChild']]
    rates=[]
    for k in keys:
        s=pd.Series(y[wc_tr], index=k.iloc[wc_tr].values); rates.append(s.groupby(level=0).mean())
    preds=full.loc[ap_idx,'IsWoman'].astype(int).values.copy()
    for j,i in enumerate(ap_idx):
        rate=None
        for k,r in zip(keys,rates):
            v=k.iat[i]
            if v in r.index: rate=r[v]; break
        if rate is None: continue
        if full.at[i,'IsWoman'] and rate==0: preds[j]=0
        if full.at[i,'IsBoy'] and rate==1: preds[j]=1
    return preds

## Families for grouped cross-validation
Passengers are linked if they share a ticket or a surname and class, and linked passengers form one family (connected components). Grouped CV never splits a family across training and validation.

In [5]:
# connected families for grouped CV: link passengers sharing a ticket or a surname+class
parent=list(range(len(full)))
def find(a):
    while parent[a]!=a: parent[a]=parent[parent[a]]; a=parent[a]
    return a
for col in [full['Ticket'], full['Surname']+'-'+full['Pclass'].astype(str)]:
    for _,idx in col.groupby(col).groups.items():
        idx=list(idx)
        for b in idx[1:]: parent[find(b)]=find(idx[0])
family_id=np.array([find(i) for i in range(len(full))])

fid = family_id[:n]
print("Families in training data:", len(set(fid)), "| largest family:", pd.Series(fid).value_counts().max())

Families in training data: 623 | largest family: 9


## Stratified vs grouped cross-validation
Both use 5 folds × 5 repeats = 25 splits.

In [6]:
idx_tr = np.arange(n)
splits_s = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42).split(idx_tr, y))
splits_g = []
for rep in range(5):
    splits_g += list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=rep).split(idx_tr, y, groups=fid))

def gender_acc(b): return (full.loc[b, 'IsWoman'].astype(int).values == y[b]).mean()
rows = [{'Family definition': 'Gender rule (no families)',
         'Stratified CV %': np.mean([gender_acc(b) for a, b in splits_s]) * 100,
         'Grouped CV %': np.mean([gender_acc(b) for a, b in splits_g]) * 100}]
for name, keys in DEFS.items():
    rows.append({'Family definition': name,
                 'Stratified CV %': np.mean([(rule(keys, a, b) == y[b]).mean() for a, b in splits_s]) * 100,
                 'Grouped CV %': np.mean([(rule(keys, a, b) == y[b]).mean() for a, b in splits_g]) * 100})
cv_table = pd.DataFrame(rows)
cv_table['Gain vs gender (stratified)'] = cv_table['Stratified CV %'] - cv_table.loc[0, 'Stratified CV %']
cv_table['Gain vs gender (grouped)'] = cv_table['Grouped CV %'] - cv_table.loc[0, 'Grouped CV %']
cv_table.round(2)

,Family definition,Stratified CV %,Grouped CV %,Gain vs gender (stratified),Gain vs gender (grouped)
0,Gender rule (no families),78.68,78.67,0.00,0.0
1,"Current (surname+class+ticket[:-1]+port+fare, ...",84.13,78.67,5.45,0.0
2,Drop last digit only (not last character),84.13,78.67,5.45,0.0
3,Without fare,84.13,78.67,5.45,0.0
4,Without fare or port,84.13,78.67,5.45,0.0
5,"Surname + exact ticket, ticket fallback",83.77,78.67,5.09,0.0
6,Exact ticket only,83.73,78.67,5.05,0.0
7,"Current, no ticket fallback",84.15,78.67,5.48,0.0
8,Surname + class only (loose),83.66,78.67,4.98,0.0


## Test-set stability
How many test passengers each definition changes from the gender rule, and how many of those are among the 17 changed by the submitted file.

In [7]:
te_idx = np.arange(n, len(full))
base = full.loc[te_idx, 'IsWoman'].astype(int).values
submitted = rule(DEFS[list(DEFS)[0]], idx_tr, te_idx)
submitted_changed = set(np.where(submitted != base)[0])
rows = []
for name, keys in DEFS.items():
    p = rule(keys, idx_tr, te_idx); ch = set(np.where(p != base)[0])
    rows.append({'Family definition': name, 'Test changes vs gender': len(ch),
                 'Shared with submitted 17': len(ch & submitted_changed),
                 'Differs from submitted file': int((p != submitted).sum())})
pd.DataFrame(rows)

,Family definition,Test changes vs gender,Shared with submitted 17,Differs from submitted file
0,"Current (surname+class+ticket[:-1]+port+fare, ...",17,17,0
1,Drop last digit only (not last character),17,17,0
2,Without fare,17,17,0
3,Without fare or port,17,17,0
4,"Surname + exact ticket, ticket fallback",15,15,2
5,Exact ticket only,15,15,2
6,"Current, no ticket fallback",16,16,1
7,Surname + class only (loose),17,16,2


In [8]:
train_families = set(family_id[:n])
has_family = np.array([family_id[i] in train_families for i in te_idx])
wc = full.loc[te_idx, 'WomanOrChild'].values
print("Test passengers with a relative in the training data:", has_family.sum(), "of", len(te_idx))
print("Test women and boys with a relative in the training data:", (has_family & wc).sum(), "of", wc.sum())

Test passengers with a relative in the training data: 206 of 418
Test women and boys with a relative in the training data: 114 of 173


## Findings
1. **The family definition barely matters.** Every reasonable definition scores 83.7–84.2% in stratified CV. Trimming the ticket's last *digit* instead of last *character*, dropping fare, or dropping port gives exactly the same results as the submitted definition.
2. **The flagged data quirks are tiny.** Only 4 tickets end in a letter (all "LINE"), 2 passengers lack a port (a maid and her employer on one ticket, both in the training data), and 1 passenger lacks a fare (an adult man, whom the rule never changes).
3. **The test-set changes are stable.** All definitions change 15–17 test passengers, and nearly all are among the submitted 17. The submitted file is not an artifact of one grouping choice.
4. **On unseen families, the rule adds nothing.** Under grouped CV it scores exactly the gender rule, because it can only use relatives already in the training data. That is by design, not a flaw.
5. **So what does 0.80143 measure?** Kaggle's test set shares families with the training set: about half of the test passengers have a relative in the training data. The rule exploits that overlap, which is legitimate for this competition, but it would not help for a new ship with no known relatives. For that question, the gender-and-class models are the relevant baseline.